### Zadanie 2 - MongoDB

Z API GeckoTerminal (`https://api.geckoterminal.com/api/v2/networks`) pobierz informacje o dostepnych sieciach kryptowalutowych. Zapisz je jako dokumenty w kolekcji MongoDB.

Uzyj agregacji, zeby policzyc liczbe sieci per typ.

*Tip: MongoDB akceptuje dict bezposrednio - zadnej parametryzacji jak w SQL.*

**Wymaga dzialajacego serwera MongoDB** (lokalnie, Docker lub Atlas).

In [2]:
from pymongo import MongoClient
import requests

# 1. Polacz z MongoDB
client = MongoClient("mongodb://localhost:27017")
db = client.lab4
networks = db["networks"]
networks.drop()  # czyscimy poprzednie dane


ServerSelectionTimeoutError: localhost:27017: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6a12e6f888f5dc62c9d7930c, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>

In [ ]:
# 2. Pobierz dane z API
response = requests.get("https://api.geckoterminal.com/api/v2/networks")
data = response.json()["data"]
print(f"Pobrano {len(data)} sieci.")
print("Przyklad:", data[0])


In [ ]:
# 3. Wstaw dokumenty
networks.insert_many(data)
print(f"W kolekcji: {networks.count_documents({})} dokumentow")


In [ ]:
# 4. Agregacja - ile sieci per typ
pipeline = [
    {"$group": {"_id": "$type", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]
print("=== Liczba sieci per typ ===")
for doc in networks.aggregate(pipeline):
    print(f"  {doc['_id']}: {doc['count']}")


In [ ]:
# Dodatkowe zapytania
print("=== Pierwsze 10 nazw sieci ===")
for n in networks.find({}, {"_id": 0, "attributes.name": 1}).limit(10):
    print(f"  - {n['attributes']['name']}")

print("\n=== Sieci zaczynajace sie od 'E' ($regex) ===")
for n in networks.find({"attributes.name": {"$regex": "^E"}}, {"_id": 0, "attributes.name": 1}):
    print(f"  - {n['attributes']['name']}")

client.close()
